# A Simple RAG Demo Project

Before running the notebook, follow the setup instructions in [README.md](README.md).

In [ ]:
# Run once when setting up the project in Colab
#!git clone https://github.com/raivisskadins/simple-rag-demo.git
#%cd simple-rag-demo
#!git pull

# Run once if packages are missing (required also for Colab)
#!pip install -r requirements.txt

# Run to pull latest changes from GitHub
#!git pull

In [ ]:
# ==============================
# 1. Imports
# ==============================

import os
import sys
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from groq import Groq
from dotenv import load_dotenv

In [ ]:
# ==============================
# 2. Load environment variables
# ==============================

load_dotenv()

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

In [ ]:
# ==============================
# 3. Load knowledge base
# ==============================

kb_files = [
    "./data/knowledge.txt",
    "./data/python_faq.txt",
    "./data/riga_facts.txt",
]

chunks = []
for filepath in kb_files:
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
    chunks.extend([p.strip() for p in text.split("\n\n") if p.strip()])

print("Chunks:")
for c in chunks:
    print("-", c)

In [ ]:
# ==============================
# 4. Create embeddings
# ==============================

def get_embedding(text):
    emb = embedding_model.encode(text)
    emb = emb / np.linalg.norm(emb)
    return emb

chunk_embeddings = np.array(
    [get_embedding(chunk) for chunk in chunks]
).astype("float32")
embedding_dim = chunk_embeddings.shape[1]



# ==============================
# 5. Create FAISS index
# ==============================

index = faiss.IndexFlatIP(embedding_dim)
# remove previously added content
index.reset()
index.add(chunk_embeddings)

print("FAISS index size:", index.ntotal)

In [ ]:
# ==============================
# 6. User question
# ==============================

#question = "Which ocean is the largest on Earth?"
question = "Is Great Wall of China visible from the Moon?"

print("Question:", question)


# ==============================
# 7. Embed the question
# ==============================

question_embedding = np.array(
    [get_embedding(question)]
).astype("float32")

# You can print out the embedding just in case you want to see how it looks
#print("Question Embedding:", question_embedding)


In [ ]:
# ==============================
# 8. Retrieve top 2 chunks
# ==============================

k = 2

distances, indices = index.search(question_embedding, k)

retrieved_chunks = [chunks[i] for i in indices[0]]

print("Retrieved context:")
for c in retrieved_chunks:
    print("-", c)

In [ ]:
# ==============================
# 9. Create RAG prompt
# ==============================

context = "\n\n".join(retrieved_chunks)

prompt = f"""
# Instructions
Answer the following question in one short response.
Use only the context provided, even if it contains false facts.
Say "There is no information awailable" if there is no answer in the context

# Context:
{context}

# Question:
{question}

# Answer:
"""

#print("Prompt:", prompt)

In [ ]:
# ==============================
# 10. Call Groq LLM to generate the answer
# ==============================

response = client.chat.completions.create(
    model="llama3-8b-8192",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

print("\nLLM Answer:")
print(response.choices[0].message.content)

In [ ]:
# ==============================
# 11. Test questions
# ==============================

test_questions = [
    "What is a Python virtual environment and how do you create one?",
    "What is the population of Riga and what river flows through it?",
    "How do you install a Python package using pip?",
]

for test_q in test_questions:
    q_emb = np.array([get_embedding(test_q)]).astype("float32")
    _, idxs = index.search(q_emb, 2)
    ctx = "\n\n".join([chunks[i] for i in idxs[0]])

    test_prompt = f"""
# Instructions
Answer the following question in one short response.
Use only the context provided, even if it contains false facts.
Say "There is no information available" if there is no answer in the context

# Context:
{ctx}

# Question:
{test_q}

# Answer:
"""

    resp = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[{"role": "user", "content": test_prompt}],
        temperature=0
    )

    print(f"Q: {test_q}")
    print(f"A: {resp.choices[0].message.content}")
    print()